# Kapitel 16

In [1]:
(lambda x: x**2)(5)

25

In [2]:
Id=lambda x: x

In [3]:
addiere= lambda x: lambda y: x+y
addiere(5)

<function __main__.<lambda>.<locals>.<lambda>(y)>

In [4]:
addiere(5)(3)

8

In [5]:
Maximum=lambda x: lambda y: x if x>y else y
[Maximum(2)(3),Maximum(3)(2)] 

[3, 3]

In [6]:
NichtNeg=Maximum(0)
[NichtNeg(x) for x in range(-2,3)]

[0, 0, 0, 1, 2]

In [7]:
fak=lambda n: 1 if n<=1 else n*fak(n-1)
fak(5)

120

In [8]:
# Y Fixpunkt-Kombinator 
Y = lambda f: (lambda x: f(lambda v: x(x)(v)))(lambda x: f(lambda v: x(x)(v)))
fak = Y(lambda rec: (lambda n: 1 if n<=1 else n*rec(n-1)))
for i in range(6):
    print(i, fak(i))

0 1
1 1
2 2
3 6
4 24
5 120


In [9]:
LATrue= lambda x,y: x 
LAFalse= lambda x,y: y 
LAAnd=lambda x,y:x(y,x) 
LAOr=lambda x,y:x(x,y) 
LANot=lambda x:x(LAFalse,LATrue) 
LAIf=lambda p,x,y:p(x,y)
print("LAIf(LAFalse,2,3)=",LAIf(LAFalse,2,3))
def asString(a): # Rückverwandlung in gewöhnlichen String     
    if a==LATrue: return "TRUE"     
    if a==LAFalse: return "FALSE"
print("Test mit de-Morgans Gesetz: not(a or b)= not(a) and not(b)")
for a in [LAFalse,LATrue]:     
    for b in [LAFalse,LATrue]:         
        L=LANot(LAOr(a,b))         
        R=LAAnd(LANot(a),LANot(b))         
        print("a=",asString(a)," b=",asString(b), " not(a or b)=",asString(L),
           " not(a) and not(b)=",asString(R))

LAIf(LAFalse,2,3)= 3
Test mit de-Morgans Gesetz: not(a or b)= not(a) and not(b)
a= FALSE  b= FALSE  not(a or b)= TRUE  not(a) and not(b)= TRUE
a= FALSE  b= TRUE  not(a or b)= FALSE  not(a) and not(b)= FALSE
a= TRUE  b= FALSE  not(a or b)= FALSE  not(a) and not(b)= FALSE
a= TRUE  b= TRUE  not(a or b)= FALSE  not(a) and not(b)= FALSE


In [10]:
# Zahlen
def P2L(n:int): # Verwandelt Python-Zahl in Lambda-Zahl 
    if n==0: 
        return LAZero 
    return LASucc(P2L(n-1)) 
def L2P(num)->int: # Wandelt Lambda-Zahl -> Python 
    return num(lambda x: x+1)(0)
LAZero=lambda f: lambda x: x 
LASucc=lambda num: lambda f: lambda x: f(num(f)(x)) 
LAAdd=lambda a,b: a(LASucc)(b) 
LAMul=lambda a,b: lambda f: a(b(f)) 
LAPot=lambda a,b: b(a)
eins=LASucc(LAZero) 
zwei=LASucc(eins) 
drei=LASucc(zwei) 
vier=LASucc(drei)
print("3=",L2P(drei)) 
print("2+3=",L2P(LAAdd(zwei,drei))) 
print("2*3=",L2P(LAMul(zwei,drei))) 
print("2^3=",L2P(LAPot(zwei,drei)))

3= 3
2+3= 5
2*3= 6
2^3= 8


In [11]:
# Tupel
LAPair = lambda a: lambda b: lambda sel: sel(a)(b)
LAFst  = lambda p: p(lambda a: lambda b: a)
LASnd  = lambda p: p(lambda a: lambda b: b)
p = LAPair(zwei)(drei)        # (2,3)
print("fst =", L2P(LAFst(p))) # 2
print("snd =", L2P(LASnd(p))) # 3

fst = 2
snd = 3


In [12]:
# Subtraktion mit phi
LAphi   = lambda p: LAPair(LASnd(p))(LASucc(LASnd(p)))
LAPred  = lambda n: LAFst(n(LAphi)(LAPair(LAZero)(LAZero)))
LAMinus = lambda a,b: b(LAPred)(a)
print("pred 0 =", L2P(LAPred(LAZero)))    # 0
print("pred 3 =", L2P(LAPred(drei)))      # 2
print("3-2   =", L2P(LAMinus(drei,zwei))) # 1
print("3-4   =", L2P(LAMinus(drei,vier))) # 0

pred 0 = 0
pred 3 = 2
3-2   = 1
3-4   = 0


In [13]:
# Listen. LANil=leere Liste; cons(1,[2,3,4]) -> [1,2,3,4]
LANil  = lambda c: lambda n: n
LACons = lambda h: lambda t: (lambda c: lambda n: c(h)(t(c)(n)))
# Länge als Church-Zahl
LALen = lambda l: l(lambda _h: lambda acc: LASucc(acc))(LAZero)
# Map(f,L)=[f(x) for x in L] 
LAMap = lambda f: lambda l: l(lambda h: lambda acc: LACons(f(h))(acc))(LANil)
# isNil → Python-bool (ohne Church-Bools)
LAIsNil = lambda l: l(lambda _h: lambda _acc: False)(True)
# Verwandeln Python Listen <-> Lambda Listen
def PList2L(lst, enc=lambda x: x):
    result = LANil
    for x in reversed(lst):
        result = LACons(enc(x))(result)
    return result
def LList2P(ll, dec=lambda x: x):
    return ll(lambda h: lambda acc: [dec(h)] + acc)([])
# Church-Liste [1,2,3]
lst123 = PList2L([eins, zwei, drei])
print("isNil([]):", LAIsNil(LANil))           # True
print("isNil([1,2,3]):", LAIsNil(lst123))     # False
print("len([1,2,3]) =", L2P(LALen(lst123)))   # 3
print("to Python:", LList2P(lst123, L2P))     # [1, 2, 3]
# map: n ↦ n+1
plus1 = lambda n: LAAdd(n, eins)
lst_plus1 = LAMap(plus1)(lst123)
print("map(+1):", LList2P(lst_plus1, L2P))    # [2, 3, 4]

isNil([]): True
isNil([1,2,3]): False
len([1,2,3]) = 3
to Python: [1, 2, 3]
map(+1): [2, 3, 4]
